<a href="https://www.kaggle.com/code/samithsachidanandan/training-a-feed-forward-network?scriptVersionId=309226525" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Import Libraries 

In [1]:
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets 
from torchvision.transforms import ToTensor 

In [2]:
BARCH_SIZE = 128 
EPOCHS = 10 
LEARNING_RATE = 0.001

In [3]:
class FeedForwardNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten =nn.Flatten()
        self.dense_layers = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, input_data):
        x = self.flatten(input_data)
        x = self.dense_layers(x)

        return x 
     

# Download Dataset 

In [4]:
def download_mnist_datasets():
    train_data = datasets.MNIST(
        root = "/kaggle/working/",
        download = True,
        train = True, 
        transform = ToTensor()
       
    )

    validation_data = datasets.MNIST(
        root = "/kaggle/working/",
        download = True,
        train = False, 
        transform = ToTensor()
       
    )
    return train_data, validation_data

In [5]:
train_data, _ = download_mnist_datasets()


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.1MB/s]


In [6]:

train_data_loader = DataLoader(train_data, batch_size = BARCH_SIZE)

In [7]:
def train_one_epoch(model, data_loader, loss_fn, optimiser, device):
    for inputs, targets in data_loader: 
        inputs, targets = inputs.to(device), targets.to(device)

        predictions = model(inputs)
        loss = loss_fn(predictions, targets)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        
    print(F"Loss: {loss.item()}") 
        

In [8]:
def train(model, data_loader, loss_fn, optimiser, device, epochs):
    for i in range(epochs):
        print(f"Epoch {i+1}")
        train_one_epoch(model, data_loader, loss_fn, optimiser, device)
        print("----------------")
    print("Training is done.")

In [9]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using {device} device")

Using cuda device


In [10]:
feed_forward_net = FeedForwardNet().to(device)



# Initialise loss funtion + optimiser

In [11]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(feed_forward_net.parameters(), lr=LEARNING_RATE)

# Train Model

In [12]:
train(feed_forward_net, train_data_loader, loss_fn,optimiser, device, EPOCHS )

Epoch 1
Loss: 0.26455190777778625
----------------
Epoch 2
Loss: 0.2127590924501419
----------------
Epoch 3
Loss: 0.18786919116973877
----------------
Epoch 4
Loss: 0.17298342287540436
----------------
Epoch 5
Loss: 0.15986564755439758
----------------
Epoch 6
Loss: 0.14380280673503876
----------------
Epoch 7
Loss: 0.12711596488952637
----------------
Epoch 8
Loss: 0.1066109761595726
----------------
Epoch 9
Loss: 0.0869578942656517
----------------
Epoch 10
Loss: 0.06769917905330658
----------------
Training is done.


# Save Model 

In [13]:
torch.save(feed_forward_net.state_dict(), "feedforward.pth")
print("Model trained and stored at feedforwardnet.pth")

Model trained and stored at feedforwardnet.pth


# Loading the Model 

In [14]:
feed_forward_net = FeedForwardNet()
state_dict = torch.load('/kaggle/working/feedforward.pth')
feed_forward_net.load_state_dict(state_dict)

<All keys matched successfully>

In [15]:
_, validation_data = download_mnist_datasets()

In [16]:
input, target = validation_data[0][0], validation_data[0][1]
input = input.unsqueeze(0)

In [17]:
class_mapping = [ "0", "1", "2", "3", "4", "5", "6", "7", "8", "9" ]

In [18]:
def predict(model, input, target, class_mapping):
    model.eval()
    with torch.no_grad():
        predictions = model(input)
        
        predicted_index = predictions[0].argmax(0)

        predicted = class_mapping[predicted_index]
        expected = class_mapping[target]
        return predicted, expected 

In [19]:
predicted, expected = predict(feed_forward_net, input, target, class_mapping)

print(f"Predicted: {predicted}, expected: {expected}")

Predicted: 7, expected: 7
